# A: Detector (Iroda)

Tasks A1-A5: load FigNews, inspect labels, merge annotators, clean, split, mask sides.
Label names are printed first; the mapping below must be filled in from what is printed, not guessed.

In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%load_ext autoreload
%autoreload 2

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

## A1. Load FigNews (pinned revision) and inspect

In [ ]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset(config.FIGNEWS_DATASET, revision=config.FIGNEWS_REVISION, split="train")
df = ds.to_pandas()
print(df.shape)
print(df.columns.tolist())
print(df["subtask"].value_counts())

In [ ]:
# Pick the bias subtask name from the output above.
BIAS_SUBTASK = None  # TODO: fill in from df["subtask"].value_counts()
bias = df[df["subtask"] == BIAS_SUBTASK]
print(bias["label"].value_counts())
print(bias["source_language"].value_counts())

## A1b. Map labels to the 3 classes
Fill from the printed label names; everything not listed is dropped.

In [ ]:
LABEL_MAP = {
    # "<exact FigNews label>": "unbiased",
    # "<exact FigNews label>": "against_SIDE_A",   # biased against Israel
    # "<exact FigNews label>": "against_SIDE_B",   # biased against Palestine
}
assert set(LABEL_MAP.values()) <= set(config.LABELS)

## A2. Merge annotators (keep posts with >= 2 agreeing)

In [ ]:
mapped = bias.assign(y=bias["label"].map(LABEL_MAP)).dropna(subset=["y"])
votes = mapped.groupby("id")["y"].agg(lambda s: s.value_counts())
# TODO: keep ids whose top label has >= config.MIN_ANNOTATOR_AGREEMENT votes; print class counts

## A5. Mask side names and check for leftovers

In [ ]:
from src.placeholders import mask_sides, find_leftovers

example = "Israeli forces and Hamas fighters clashed near Gaza, Netanyahu said."
masked = mask_sides(example)
print(masked, find_leftovers(masked))